In [1]:
from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import prepare_all_splits

prepare_all_splits()

NameError: name 'List' is not defined

In [1]:
from FIT_python.plot_style import apply_style

apply_style()
import pandas as pd
from pathlib import Path
from FIT_python.pipeline_individual_id.utils import sequential_holdout_ids
from FIT_python.pipeline_individual_id.geometric_pairwise_projection import (
    generate_pairwise_comparisons_from_df,
)
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline
from FIT_python.pipeline_individual_id.supervised_umap_pipeline import run as run_umap

# from FIT_python.pipeline_individual_id.siamese_pipeline import run as run_siamese
from FIT_python.pipeline_individual_id.evaluation import compute_confusion
import seaborn as sns, matplotlib.pyplot as plt
from FIT_python.data_split_and_summary.data_import_utils import (
    load_and_prep_df_individual,
    get_feature_cols,
)
from FIT_python.config import RESULTS_DATA_DIR, SPLITS_DIR


# %% 1) Basis-Pfade & Hyperparameter-Räume (unverändert)
splits_root = Path("data/splits/eurasian_otter")
models_root = (
    RESULTS_DATA_DIR
    / "eurasian_otter_random_search_standard_metrics"
    / "best_maj_test_pct"
)
output_root = Path("results/data/pairwise_comp_eurasian_otter")
output_root.mkdir(parents=True, exist_ok=True)

# 6) Loop über Spezies
species_dir = Path(SPLITS_DIR) / "eurasian_otter"
species = "eurasian_otter"
print(f"\n🔄 Processing species: {species}")

# 6.1) Train/Test laden & Fold entfernen
df = load_and_prep_df_individual(
    species, Path(SPLITS_DIR)
)  # falls hier bugs valid vales f und m checken

feature_cols = [
    c for c in df.columns if c.startswith(("dist", "ang", "t", "v")) and c != "trail"
]
ids = df["individual_id"].unique()
split = sequential_holdout_ids(ids, val_sizes=[3, 5, 9], n_iter=1, random_state=0)[0]
train_df = df[df["individual_id"].isin(split["train_ids"])]
val_df = df[df["individual_id"].isin(split["val_ids"])]
val_comps = generate_pairwise_comparisons_from_df(
    val_df,
    id_col="individual_id",
    group_sizes=[  7],
    n_repeats=2,
    mode="both",
    selfmatch_factor=2.0,
    n_folds=3,
    random_state=42,
    show_progress=True,
)

# In ein DataFrame überführen
df_val_comps = pd.DataFrame(val_comps)

c:\Users\Frede\miniconda3\envs\FIT_python_conda_env\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm



🔄 Processing species: eurasian_otter


c:\Users\Frede\miniconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


In [2]:
val_comps

[{'ind_a': 'leni',
  'ind_b': 'zewe',
  'samples_a': ['Eurasian_Otter_262',
   'Eurasian_Otter_266',
   'Eurasian_Otter_261',
   'Eurasian_Otter_270',
   'Eurasian_Otter_267',
   'Eurasian_Otter_258',
   'Eurasian_Otter_264'],
  'samples_b': ['Eurasian_Otter_349',
   'Eurasian_Otter_350',
   'Eurasian_Otter_353',
   'Eurasian_Otter_358',
   'Eurasian_Otter_352',
   'Eurasian_Otter_351',
   'Eurasian_Otter_359'],
  'trail_a_id': 'leni_7a',
  'trail_b_id': 'zewe_7a',
  'same_individual': False,
  'fold': 0},
 {'ind_a': 'leni',
  'ind_b': 'zewe',
  'samples_a': ['Eurasian_Otter_258',
   'Eurasian_Otter_269',
   'Eurasian_Otter_265',
   'Eurasian_Otter_263',
   'Eurasian_Otter_266',
   'Eurasian_Otter_262',
   'Eurasian_Otter_261'],
  'samples_b': ['Eurasian_Otter_355',
   'Eurasian_Otter_359',
   'Eurasian_Otter_364',
   'Eurasian_Otter_350',
   'Eurasian_Otter_356',
   'Eurasian_Otter_357',
   'Eurasian_Otter_361'],
  'trail_a_id': 'leni_7b',
  'trail_b_id': 'zewe_7b',
  'same_individual

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline
from FIT_python.pipeline_individual_id.evaluation import compute_confusion

# 1) Generiere die Features & Labels
res_base = DistanceBaseline.run(train_df, val_comps, feature_cols)
Xb = res_base.filter(regex="^dist_")
yb = res_base["same_individual"].map({"True": 1, "False": 0})

# 2) Definiere Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
lr = LogisticRegression(max_iter=200)

# 3) Erzeuge CV-Predictions für jedes Sample im Validation-Set
y_pred = cross_val_predict(lr, Xb, yb, cv=cv)

# 4) Berechne die Konfusionsmatrix
cm_base = compute_confusion(pd.DataFrame({"same_individual": yb, "pred": y_pred}))

# 5) Visualisiere
plt.figure(figsize=(6, 5))
sns.heatmap(cm_base, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("5-Fold Stratified CV Confusion Matrix")
plt.show()

Processing Pairs:   0%|          | 0/14 [00:00<?, ?it/s]

  0%|          | 0/14 [00:00<?, ?it/s]

KeyError: "None of [Index(['Eurasian_Otter_448', 'Eurasian_Otter_454', 'Eurasian_Otter_451',\n       'Eurasian_Otter_449', 'Eurasian_Otter_450', 'Eurasian_Otter_453',\n       'Eurasian_Otter_452'],\n      dtype='object', name='id')] are in the [index]"

In [ ]:
res_umap, emb_umap = run_umap(train_df, val_comps, feature_cols)
yu = res_umap["same_individual"].map({"True": 1, "False": 0})
pred_u = (res_umap["pred_same_proba"] > 0.5).astype(int)
cm_umap = compute_confusion(pd.DataFrame({"same_individual": yu, "pred": pred_u}))
sns.heatmap(cm_umap, annot=True, fmt="d")
plt.show()

c:\Users\Frede\miniconda3\envs\FIT_python_conda_env\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
Processing Pairs:   0%|          | 0/14 [01:52<?, ?it/s]


KeyError: "None of [Index(['214', '211', '221', '215', '209', '220', '219'], dtype='object', name='id')] are in the [index]"

In [ ]:
res_siam = run_siamese(train_df, val_comps, feature_cols)
ys = pd.Series([c["same_individual"] for c in val_comps]).map({"True": 1, "False": 0})
pred_s = (pd.Series([r["pred_same"] for r in res_siam]) > 0.5).astype(int)
cm_siam = compute_confusion(pd.DataFrame({"same_individual": ys, "pred": pred_s}))
sns.heatmap(cm_siam, annot=True, fmt="d")
plt.show()

NameError: name 'run_siamese' is not defined